In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

In [3]:
DATA_PATH = Path("../data/raw/bank.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [4]:
target_col = "deposit"

y = df[target_col].map({
    "no": 0,
    "yes": 1,
})

target_distribution = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "proportion": y.value_counts(normalize=True).sort_index(),
})

target_distribution

,count,proportion
deposit,,
0,5873,0.5262
1,5289,0.4738


In [5]:
excluded_cols = [
    "deposit",
    "duration",
]

X = df.drop(columns=excluded_cols)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExcluded columns:")
print(excluded_cols)

print("\nIs deposit in X?", "deposit" in X.columns)
print("Is duration in X?", "duration" in X.columns)

X shape: (11162, 15)
y shape: (11162,)

Excluded columns:
['deposit', 'duration']

Is deposit in X? False
Is duration in X? False


In [6]:
numeric_features = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
]

expected_features = numeric_features + categorical_features

print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))
print("Total expected features:", len(expected_features))
print("Actual X features:", X.shape[1])

print("\nMissing from X:")
print(set(expected_features) - set(X.columns))

print("\nUnexpected in X:")
print(set(X.columns) - set(expected_features))

Number of numeric features: 6
Number of categorical features: 9
Total expected features: 15
Actual X features: 15

Missing from X:
set()

Unexpected in X:
set()


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8929, 15)
X_test: (2233, 15)
y_train: (8929,)
y_test: (2233,)


In [8]:
def target_distribution_by_split(series, split_name):
    counts = series.value_counts().sort_index()
    proportions = series.value_counts(normalize=True).sort_index()
    
    return pd.DataFrame({
        "split": split_name,
        "class": counts.index,
        "count": counts.values,
        "proportion": proportions.values,
    })

In [9]:
target_distribution_table = pd.concat(
    [
        target_distribution_by_split(y, "full"),
        target_distribution_by_split(y_train, "train"),
        target_distribution_by_split(y_test, "test"),
    ],
    ignore_index=True,
)

target_distribution_table

,split,class,count,proportion
0,full,0,5873,0.5262
1,full,1,5289,0.4738
2,train,0,4698,0.5262
3,train,1,4231,0.4738
4,test,0,1175,0.5262
5,test,1,1058,0.4738


In [10]:
final_model_name = "HistGradientBoostingClassifier"
final_threshold = 0.35

print("Final frozen model:", final_model_name)
print("Final frozen threshold:", final_threshold)

Final frozen model: HistGradientBoostingClassifier
Final frozen threshold: 0.35


In [11]:
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_features),
        ("cat", categorical_preprocessor, categorical_features),
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [12]:
final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingClassifier(random_state=42)),
    ]
)

final_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## Stage 6 — Final evaluation setup

Task type: binary classification.

Target: `deposit`.

Positive class: `yes`, encoded as `1`.

Main scenario: pre-contact prediction.

Final frozen candidate:

- `HistGradientBoostingClassifier(random_state=42)`

Final frozen threshold policy:

- predict `1` if predicted probability >= `0.35`
- otherwise predict `0`

Data preparation:

- `y = deposit`, with `no -> 0`, `yes -> 1`
- `X = df.drop(columns=["deposit", "duration"])`
- `duration` excluded due to leakage risk
- `unknown` values kept as categorical levels

Split:

- `test_size=0.2`
- `random_state=42`
- `stratify=y`

Important final-evaluation rules:

- This is the first and only evaluation on `X_test`.
- No threshold changes after seeing test results.
- No model parameter changes after seeing test results.
- No alternative models evaluated on `X_test`.
- Final model is not saved yet at this stage.

In [13]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [14]:
final_pipeline.fit(X_train, y_train)

print("Final pipeline fitted on X_train / y_train only.")

Final pipeline fitted on X_train / y_train only.


In [15]:
y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]

print("Predicted probabilities on X_test generated once.")
print("Number of test probabilities:", len(y_test_proba))

Predicted probabilities on X_test generated once.
Number of test probabilities: 2233


In [16]:
y_test_pred = (y_test_proba >= final_threshold).astype(int)

print("Frozen threshold applied:", final_threshold)
print("Predicted class distribution:")
print(pd.Series(y_test_pred).value_counts().sort_index())
print(pd.Series(y_test_pred).value_counts(normalize=True).sort_index())

Frozen threshold applied: 0.35
Predicted class distribution:
0     993
1    1240
Name: count, dtype: int64
0   0.4447
1   0.5553
Name: proportion, dtype: float64


In [17]:
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, y_test_proba)
test_pr_auc = average_precision_score(y_test, y_test_proba)

final_test_metrics = pd.DataFrame(
    [
        {
            "model": final_model_name,
            "threshold": final_threshold,
            "accuracy": test_accuracy,
            "precision": test_precision,
            "recall": test_recall,
            "f1": test_f1,
            "roc_auc": test_roc_auc,
            "pr_auc": test_pr_auc,
        }
    ]
)

final_test_metrics_rounded = final_test_metrics.copy()

metric_cols = [
    "threshold",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

final_test_metrics_rounded[metric_cols] = (
    final_test_metrics_rounded[metric_cols].round(4)
)

final_test_metrics_rounded

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,HistGradientBoostingClassifier,0.3500,0.7035,0.6597,0.7732,0.7119,0.7890,0.7900


In [18]:
cm = confusion_matrix(y_test, y_test_pred)

tn, fp, fn, tp = cm.ravel()

confusion_matrix_df = pd.DataFrame(
    cm,
    index=["actual_0_no", "actual_1_yes"],
    columns=["pred_0_no", "pred_1_yes"],
)

confusion_matrix_df

,pred_0_no,pred_1_yes
actual_0_no,753,422
actual_1_yes,240,818


In [19]:
confusion_components = pd.DataFrame(
    [
        {
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        }
    ]
)

confusion_components

,tn,fp,fn,tp
0,753,422,240,818


In [20]:
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["no", "yes"],
        digits=4,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

          no     0.7583    0.6409    0.6946      1175
         yes     0.6597    0.7732    0.7119      1058

    accuracy                         0.7035      2233
   macro avg     0.7090    0.7070    0.7033      2233
weighted avg     0.7116    0.7035    0.7028      2233



Ячейка — compare with Stage 5 OOF policy

In [21]:
stage5_oof_metrics = pd.DataFrame(
    [
        {
            "source": "Stage 5 OOF",
            "model": "HistGradientBoostingClassifier",
            "threshold": 0.35,
            "accuracy": 0.6972,
            "precision": 0.6515,
            "recall": 0.7762,
            "f1": 0.7084,
        },
        {
            "source": "Stage 6 test",
            "model": final_model_name,
            "threshold": final_threshold,
            "accuracy": test_accuracy,
            "precision": test_precision,
            "recall": test_recall,
            "f1": test_f1,
        },
    ]
)

stage5_vs_test = stage5_oof_metrics.copy()

for col in ["threshold", "accuracy", "precision", "recall", "f1"]:
    stage5_vs_test[col] = stage5_vs_test[col].round(4)

stage5_vs_test

,source,model,threshold,accuracy,precision,recall,f1
0,Stage 5 OOF,HistGradientBoostingClassifier,0.3500,0.6972,0.6515,0.7762,0.7084
1,Stage 6 test,HistGradientBoostingClassifier,0.3500,0.7035,0.6597,0.7732,0.7119


Ячейка — deltas vs Stage 5 OOF

In [22]:
test_vs_oof_delta = pd.DataFrame(
    [
        {
            "metric": "accuracy",
            "stage5_oof": 0.6972,
            "stage6_test": test_accuracy,
            "delta_test_minus_oof": test_accuracy - 0.6972,
        },
        {
            "metric": "precision",
            "stage5_oof": 0.6515,
            "stage6_test": test_precision,
            "delta_test_minus_oof": test_precision - 0.6515,
        },
        {
            "metric": "recall",
            "stage5_oof": 0.7762,
            "stage6_test": test_recall,
            "delta_test_minus_oof": test_recall - 0.7762,
        },
        {
            "metric": "f1",
            "stage5_oof": 0.7084,
            "stage6_test": test_f1,
            "delta_test_minus_oof": test_f1 - 0.7084,
        },
    ]
)

test_vs_oof_delta[["stage5_oof", "stage6_test", "delta_test_minus_oof"]] = (
    test_vs_oof_delta[["stage5_oof", "stage6_test", "delta_test_minus_oof"]]
    .round(4)
)

test_vs_oof_delta

,metric,stage5_oof,stage6_test,delta_test_minus_oof
0,accuracy,0.6972,0.7035,0.0063
1,precision,0.6515,0.6597,0.0082
2,recall,0.7762,0.7732,-0.0030
3,f1,0.7084,0.7119,0.0035


## Stage 6 — Final holdout evaluation conclusions

### Setup

Task type: binary classification.

Target: `deposit`.

Positive class: `yes`, encoded as `1`.

Main scenario: pre-contact prediction.

Final frozen candidate:

- `HistGradientBoostingClassifier(random_state=42)`

Final frozen threshold policy:

- predict `1` if predicted probability >= `0.35`
- otherwise predict `0`

### Data preparation

The same preparation logic from previous stages was reused:

- `y = deposit`, with `no -> 0`, `yes -> 1`
- `X = df.drop(columns=["deposit", "duration"])`
- `duration` excluded due to leakage risk
- `unknown` values kept as categorical levels
- same stratified split:
  - `test_size=0.2`
  - `random_state=42`
  - `stratify=y`

The final pipeline was fitted only on `X_train` / `y_train`.

`X_test` was evaluated once for final holdout metrics.

No alternative models were evaluated on `X_test`.

No threshold changes were made after seeing test results.

No model parameter changes were made after seeing test results.

The final model was not saved yet.

### Final test metrics

| model | threshold | accuracy | precision | recall | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|---:|---:|
| HistGradientBoostingClassifier | 0.35 | 0.7035 | 0.6597 | 0.7732 | 0.7119 | 0.7890 | 0.7900 |

### Confusion matrix

|  | pred_0_no | pred_1_yes |
|---|---:|---:|
| actual_0_no | 753 | 422 |
| actual_1_yes | 240 | 818 |

Confusion matrix components:

- TN = 753
- FP = 422
- FN = 240
- TP = 818

### Comparison with Stage 5 OOF policy

| source | threshold | accuracy | precision | recall | F1 |
|---|---:|---:|---:|---:|---:|
| Stage 5 OOF | 0.35 | 0.6972 | 0.6515 | 0.7762 | 0.7084 |
| Stage 6 test | 0.35 | 0.7035 | 0.6597 | 0.7732 | 0.7119 |

Test minus OOF:

| metric | delta |
|---|---:|
| accuracy | +0.0063 |
| precision | +0.0082 |
| recall | -0.0030 |
| F1 | +0.0035 |

### Interpretation

The final test metrics aligned very closely with the Stage 5 OOF expectations.

The threshold `0.35` behaved as expected:

- recall remained high;
- precision remained moderate;
- false negatives were reduced compared with a more conservative threshold policy;
- false positives increased as expected from the lower threshold.

The model quality appears stable based on the close match between OOF and test metrics.

There are no obvious signs of severe overfitting or threshold instability from the final holdout result.

### Honest model quality

The final model is useful for prioritizing likely deposit subscribers before contact.

It is not a perfect classifier.

Its main strength is recall:

- it captures a large share of potential subscribers;
- on the final test set, recall is `0.7732`.

Its main limitation is precision:

- some predicted positives will not subscribe;
- on the final test set, precision is `0.6597`.

This policy is appropriate if the business prefers catching more potential subscribers and can tolerate additional false-positive contacts.

If sales capacity is limited or false positives are expensive, a more conservative threshold would be preferable, but that threshold was not selected for this final evaluation.

### Limitations

- The threshold was chosen using train-only OOF predictions, not business cost optimization.
- There is no explicit monetary cost matrix for false positives vs false negatives.
- `day` and `month` exist, but there is no full timestamp/year, so a truly chronological validation strategy was not possible.
- `duration` was excluded correctly for pre-contact prediction, so results are lower but more realistic than a leakage-contaminated model would be.
- The dataset may not represent future campaigns if campaign strategy, customer population, or macro conditions change.

### Leakage checks

Passed:

- `duration` excluded from `X`
- `duration` excluded from `X_train`
- `duration` excluded from `X_test`
- `deposit` excluded from `X`
- `deposit` excluded from `X_train`
- `deposit` excluded from `X_test`
- `unknown` values kept as categorical levels
- preprocessing placed inside `Pipeline` / `ColumnTransformer`
- final pipeline fitted only on `X_train` / `y_train`
- `X_test` evaluated once
- frozen threshold `0.35` applied
- no threshold change after test evaluation
- no model parameter change after test evaluation
- no alternative model evaluated on `X_test`
- final model not saved yet